---
## 1. Business Problem

A B2B software company wants to identify which business accounts are likely to **upgrade their current plan**. Proactively identifying upgrade candidates allows the sales team to prioritise outreach, personalise offers, and increase revenue efficiently.

**Business Question:** Based on a company's size, usage behaviour, engagement, and financials, can we predict whether they will upgrade their account?

**Target Variable:** `Upgraded_Account` (Yes = upgraded, No = did not upgrade)

## 2. Machine Learning Problem

This is a **Binary Classification** problem. 
    
The model predicts whether a business account will upgrade — the target variable `Upgraded_Account` has two outcomes: *Yes* (upgraded) or *No* (did not upgrade).

Classification is appropriate because we are predicting a category, not a numerical value. 
    
Logistic Regression is selected as the baseline model due to its suitability for binary outcomes and interpretability.

In [1]:
import pandas as pd
# Load a CSV file
df = pd.read_excel('/Users/dorcasfaloye/Documents/Ontario Tech /Term 2/AI Programming /business_account_upgrade_prediction_dataset.xlsx')
df.head()# Show the first five rows

,Account_ID,Company_Size,Industry,Annual_Revenue,Monthly_Transactions,Current_Plan,Account_Age_Months,Support_Tickets,Sales_Contacted,Product_Usage_Score,Monthly_Fee,Training_Attended,Upgraded_Account
0,1,Small,Healthcare,4765975.0,368,Business,26,5,No,55.0,2832,Yes,No
1,2,Small,Manufacturing,5683548.0,257,Basic,75,4,No,5.0,1215,Yes,No
2,3,Enterprise,Manufacturing,1722517.0,424,Premium,99,0,No,64.0,753,No,No
3,4,Enterprise,Education,3481277.0,190,Standard,33,0,Yes,86.0,1973,No,Yes
4,5,Small,Finance,1023375.0,262,Basic,52,3,No,88.0,3676,Yes,No


Inspect Dataset 

In [2]:
# Show the first rows
df.head()
# Show the number of rows and columns
df.shape

# Show column names
df.columns
# Show data types and non-null counts
df.info()
# Summary statistics for numeric columns
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 377 entries, 0 to 376
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Account_ID            377 non-null    int64  
 1   Company_Size          370 non-null    str    
 2   Industry              370 non-null    str    
 3   Annual_Revenue        370 non-null    float64
 4   Monthly_Transactions  377 non-null    int64  
 5   Current_Plan          370 non-null    str    
 6   Account_Age_Months    377 non-null    int64  
 7   Support_Tickets       377 non-null    int64  
 8   Sales_Contacted       370 non-null    str    
 9   Product_Usage_Score   370 non-null    float64
 10  Monthly_Fee           377 non-null    int64  
 11  Training_Attended     377 non-null    str    
 12  Upgraded_Account      377 non-null    str    
dtypes: float64(2), int64(5), str(6)
memory usage: 38.4 KB


,Account_ID,Annual_Revenue,Monthly_Transactions,Account_Age_Months,Support_Tickets,Product_Usage_Score,Monthly_Fee
count,377.000000,3.700000e+02,377.000000,377.000000,377.000000,370.000000,377.000000
mean,186.265252,4.060962e+06,248.880637,60.962865,6.933687,51.102703,2408.000000
std,106.726653,2.315737e+06,140.723205,35.513763,4.266190,28.988835,1430.944912
min,1.000000,8.265600e+04,0.000000,1.000000,0.000000,0.000000,83.000000
25%,94.000000,1.902595e+06,130.000000,30.000000,4.000000,27.000000,1231.000000
50%,187.000000,4.182715e+06,248.000000,62.000000,7.000000,51.500000,2414.000000
75%,278.000000,6.017136e+06,363.000000,93.000000,10.000000,76.000000,3623.000000
max,370.000000,7.987036e+06,499.000000,119.000000,14.000000,100.000000,4999.000000


In [3]:
# Count missing values in each column
df.isnull().sum()

Account_ID              0
Company_Size            7
Industry                7
Annual_Revenue          7
Monthly_Transactions    0
Current_Plan            7
Account_Age_Months      0
Support_Tickets         0
Sales_Contacted         7
Product_Usage_Score     7
Monthly_Fee             0
Training_Attended       0
Upgraded_Account        0
dtype: int64

Clean the Data

In [12]:
# Create a copy of the original dataset before cleaning
# This helps me keep the original data unchanged
df_clean = df.copy()

# Duplicate rows may affect the model training process
print("\nNumber of duplicate rows before cleaning:")
print(df_clean.duplicated().sum())


Number of duplicate rows before cleaning:
0


In [4]:
#Fill the missing values 

# Fill missing values in the Company_Size column with the most frequent Company
# mode()[0] returns the most common value in the column
df_clean["Company_Size"] = df_clean["Company_Size"].fillna(df_clean["Company_Size"].mode()[0])

# Fill missing values in the Industry column with the most frequent Industry
# mode()[0] returns the most common value in the column
df_clean["Industry"] = df_clean["Industry"].fillna(df_clean["Industry"].mode()[0])

# Fill missing values in the Annual_Revenue column with the median Annual_Revenue
# The median is often used because it is less affected by very large or very small values
df_clean["Annual_Revenue"] = df_clean["Annual_Revenue"].fillna(df_clean["Annual_Revenue"].median())

# Fill missing values in the Current_Plan column with the most frequent Current_Plan
# mode()[0] returns the most common value in the column
df_clean["Current_Plan"] = df_clean["Current_Plan"].fillna(df_clean["Current_Plan"].mode()[0])

# Fill missing values in the Sales_Contacted with the most frequent Sales_Contacted
# mode()[0] returns the most common value in the column
df_clean["Sales_Contacted"] = df_clean["Sales_Contacted"].fillna(df_clean["Sales_Contacted"].mode()[0])

# Fill missing values in the Product_Usage_Score column with the median Product_Usage_Score
# The median is often used because it is less affected by very large or very small values
df_clean["Product_Usage_Score"] = df_clean["Product_Usage_Score"].fillna(df_clean["Product_Usage_Score"].median())

# Display the dataset after filling missing values
df_clean.head()

,Account_ID,Company_Size,Industry,Annual_Revenue,Monthly_Transactions,Current_Plan,Account_Age_Months,Support_Tickets,Sales_Contacted,Product_Usage_Score,Monthly_Fee,Training_Attended,Upgraded_Account
0,1,Small,Healthcare,4765975.0,368,Business,26,5,No,55.0,2832,Yes,No
1,2,Small,Manufacturing,5683548.0,257,Basic,75,4,No,5.0,1215,Yes,No
2,3,Enterprise,Manufacturing,1722517.0,424,Premium,99,0,No,64.0,753,No,No
3,4,Enterprise,Education,3481277.0,190,Standard,33,0,Yes,86.0,1973,No,Yes
4,5,Small,Finance,1023375.0,262,Basic,52,3,No,88.0,3676,Yes,No


In [5]:
# Check missing values again after cleaning

print("Missing values after cleaning:")
print(df_clean.isnull().sum())

Missing values after cleaning:
Account_ID              0
Company_Size            0
Industry                0
Annual_Revenue          0
Monthly_Transactions    0
Current_Plan            0
Account_Age_Months      0
Support_Tickets         0
Sales_Contacted         0
Product_Usage_Score     0
Monthly_Fee             0
Training_Attended       0
Upgraded_Account        0
dtype: int64


In [118]:
# Define the feature columns

feature_columns = [
    "Company_Size",
    "Industry",
    "Annual_Revenue",
    "Monthly_Transactions",
    "Current_Plan",
    "Account_Age_Months",
    "Support_Tickets",
    "Sales_Contacted",
    "Product_Usage_Score",
    "Monthly_Fee",
    "Training_Attended",
]
    
# Define X as the feature matrix
# X contains all input columns
X = df[feature_columns]

# Handle missing values on X
num_cols = X.select_dtypes(include='number').columns
X[num_cols] = X[num_cols].fillna(X[num_cols].median())

cat_cols = X.select_dtypes(include='object').columns
for col in cat_cols:
    X[col] = X[col].fillna(X[col].mode()[0])

# Confirm
print('NaN in X:', X.isnull().sum().sum())


# Define y as the target variable, y contains the column we want to predict
# Convert the target variable from text labels to numbers
# "No" becomes 0 and "Yes" becomes 1
y = df["Upgraded_Account"].map({"No": 0, "Yes": 1})

# Display the first 5 rows of X
print("Features (X):")
display(X.head())

# Display the first 5 values of y
print("Target(y):")
display(y.head())

NaN in X: 0
Features (X):


/var/folders/3q/jbtzgm0d33dchkdqq3n5kxzr0000gn/T/ipykernel_86695/1561821510.py:25: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include='object').columns


,Company_Size,Industry,Annual_Revenue,Monthly_Transactions,Current_Plan,Account_Age_Months,Support_Tickets,Sales_Contacted,Product_Usage_Score,Monthly_Fee,Training_Attended
0,Small,Healthcare,4765975.0,368,Business,26,5,No,55.0,2832,Yes
1,Small,Manufacturing,5683548.0,257,Basic,75,4,No,5.0,1215,Yes
2,Enterprise,Manufacturing,1722517.0,424,Premium,99,0,No,64.0,753,No
3,Enterprise,Education,3481277.0,190,Standard,33,0,Yes,86.0,1973,No
4,Small,Finance,1023375.0,262,Basic,52,3,No,88.0,3676,Yes


Target(y):


0    0
1    0
2    0
3    1
4    0
Name: Upgraded_Account, dtype: int64

Prepocessing 

In [119]:
# Import ColumnTransformer to apply different preprocessing steps to different columns
from sklearn.compose import ColumnTransformer

# Import OneHotEncoder for categorical variables
# Import StandardScaler for numerical variables
from sklearn.preprocessing import OneHotEncoder, StandardScaler


In [121]:
# Define numerical features
# These columns contain numbers and will be scaled
numerical_features = [
    "Annual_Revenue",
    "Monthly_Transactions",
    "Account_Age_Months",
    "Support_Tickets",
    "Product_Usage_Score",
    "Monthly_Fee"
]

In [122]:
# Define categorical features
# These columns contain text categories and will be encoded
categorical_features = [
    "Industry",
    "Company_Size",
    "Current_Plan",
    "Sales_Contacted",
    "Training_Attended"
]

In [123]:
# Create a preprocessing object
# StandardScaler will be applied to numerical columns
# OneHotEncoder will be applied to categorical columns
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        #("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)

    ]
)

In [124]:
# Display the first 5 rows of the feature matrix
print("Features before preprocessing:")

display(X.head())

Features before preprocessing:


,Company_Size,Industry,Annual_Revenue,Monthly_Transactions,Current_Plan,Account_Age_Months,Support_Tickets,Sales_Contacted,Product_Usage_Score,Monthly_Fee,Training_Attended
0,Small,Healthcare,4765975.0,368,Business,26,5,No,55.0,2832,Yes
1,Small,Manufacturing,5683548.0,257,Basic,75,4,No,5.0,1215,Yes
2,Enterprise,Manufacturing,1722517.0,424,Premium,99,0,No,64.0,753,No
3,Enterprise,Education,3481277.0,190,Standard,33,0,Yes,86.0,1973,No
4,Small,Finance,1023375.0,262,Basic,52,3,No,88.0,3676,Yes


In [125]:
# Display the first 5 values of the target variable
print("Upgraded_Account after converting Yes/No to 1/0:")

display(y.head())

Upgraded_Account after converting Yes/No to 1/0:


0    0
1    0
2    0
3    1
4    0
Name: Upgraded_Account, dtype: int64

Train/Test Split 

In [126]:
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
# Display the shape of the training and testing sets

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (296, 11)
X_test shape: (74, 11)
y_train shape: (296,)
y_test shape: (74,)


In [127]:
# Display the first 5 rows of the training features

print("Training features:")
display(X_train.head())

# Display the first 5 rows of the testing features

print("Testing features:")
display(X_test.head())

Training features:


,Company_Size,Industry,Annual_Revenue,Monthly_Transactions,Current_Plan,Account_Age_Months,Support_Tickets,Sales_Contacted,Product_Usage_Score,Monthly_Fee,Training_Attended
298,Medium,Retail,1356774.0,162,Basic,62,0,No,44.0,1951,Yes
302,Small,Technology,5991368.0,497,Business,10,0,No,43.0,555,Yes
366,Small,Retail,4131182.0,346,Business,57,2,Yes,44.0,2883,No
41,Small,Education,5202171.0,46,Standard,75,3,No,68.0,3741,Yes
96,Medium,Healthcare,5962036.0,183,Standard,33,2,No,53.0,363,Yes


Testing features:


,Company_Size,Industry,Annual_Revenue,Monthly_Transactions,Current_Plan,Account_Age_Months,Support_Tickets,Sales_Contacted,Product_Usage_Score,Monthly_Fee,Training_Attended
323,Medium,Finance,5722480.0,11,Basic,13,6,No,69.0,2399,Yes
112,Medium,Technology,7080258.0,217,Standard,34,14,No,53.0,2684,Yes
213,Small,Education,7822509.0,173,Business,28,3,Yes,58.0,2191,No
147,Medium,Retail,4307828.0,55,Basic,27,10,No,85.0,2858,Yes
303,Small,Technology,180310.0,408,Basic,45,4,Yes,12.0,392,No


In [128]:
# Fit the preprocessor only on the training data

X_train_processed = preprocessor.fit_transform(X_train)

# Transform the test data using the same preprocessor

X_test_processed = preprocessor.transform(X_test)

In [129]:
# Display the shape before and after preprocessing

print("Before preprocessing:")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("\nAfter preprocessing:")
print("X_train_processed:", X_train_processed.shape)
print("X_test_processed:", X_test_processed.shape)

Before preprocessing:
X_train: (296, 11)
X_test: (74, 11)

After preprocessing:
X_train_processed: (296, 24)
X_test_processed: (74, 24)


Building Baseline Model

In [130]:
# Step 8: Building a Baseline Model
from sklearn.linear_model import LogisticRegression

In [131]:
# Create a Logistic Regression model
# max_iter is increased to make sure the model has enough iterations to train properly

baseline_model = LogisticRegression(max_iter=1000)

In [132]:
# Train the baseline model using the processed training data
baseline_model.fit(X_train_processed, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [133]:
# Use the trained model to make predictions on the processed test data

y_pred = baseline_model.predict(X_test_processed)

In [134]:
# Display the predicted values
# 0 means the customer is predicted not to purchase
# 1 means the customer is predicted to purchase

print("Predicted values:")
print(y_pred)

Predicted values:
[0 0 0 0 0 0 1 0 1 0 1 0 0 0 0 0 1 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 1
 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0]
